# Phase 3 — Deep Learning Modeling

This notebook trains and evaluates genre-classification models using the harmonic feature vectors extracted in Phase 1.

**Two architectures are demonstrated:**
1. `GenreClassifier` — MLP over the 19-dimensional harmonic feature vector
2. `ChordTransformer` — Transformer over chord token sequences

> **Kernel:** select **SONATAM (.venv)** in the kernel picker (top-right).


In [ ]:
# ── Imports & config ──────────────────────────────────────────────────────
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt
import torch

from sonata.config.settings             import CFG
from sonata.models                      import HarmonicDataset
from sonata.models.architectures        import GenreClassifier, ChordTransformer
from sonata.models.train                import Trainer, TrainerConfig
from sonata.models.evaluate             import classification_report_df, confusion_matrix_plot
from sonata.notebook_utils              import rp, show_paths

print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')


In [ ]:
# ── Load curated dataset ───────────────────────────────────────────────────
df = pd.read_parquet(CFG['dataset']['parquet_file'])
print(f'Dataset: {df.shape[0]:,} rows, {df.shape[1]} columns')

# Filter to genres with enough samples for stratified split
MIN_SAMPLES_PER_CLASS = 5
genre_counts = df['primary_genre'].value_counts()
valid_genres = genre_counts[genre_counts >= MIN_SAMPLES_PER_CLASS].index
df_filtered  = df[df['primary_genre'].isin(valid_genres)].copy()
print(f'After genre filter: {len(df_filtered):,} rows, {len(valid_genres)} genres')

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Model 1: MLP (GenreClassifier) — feature-vector approach
# ═══════════════════════════════════════════════════════════
dl_cfg = CFG['deep_learning']
train_cfg = dl_cfg['training']

dataset_mlp = HarmonicDataset(df_filtered, label_col='primary_genre', mode='classification')
idx_train, idx_val, idx_test = dataset_mlp.split(
    val_frac=train_cfg['val_frac'], test_frac=train_cfg['test_frac'], seed=train_cfg['seed']
)
print(f'Train: {len(idx_train)}, Val: {len(idx_val)}, Test: {len(idx_test)}')
print(f'Input dim: {dataset_mlp.input_dim}, Classes: {dataset_mlp.num_classes}')

In [ ]:
from torch.utils.data import DataLoader, Subset

train_loader = DataLoader(Subset(dataset_mlp, idx_train),
                          batch_size=train_cfg['batch_size'], shuffle=True)
val_loader   = DataLoader(Subset(dataset_mlp, idx_val),
                          batch_size=train_cfg['batch_size'], shuffle=False)
test_loader  = DataLoader(Subset(dataset_mlp, idx_test),
                          batch_size=train_cfg['batch_size'], shuffle=False)

mlp_model = GenreClassifier(
    input_dim   = dataset_mlp.input_dim,
    num_classes = dataset_mlp.num_classes,
    hidden_dims = dl_cfg['mlp']['hidden_dims'],
    dropout     = dl_cfg['mlp']['dropout'],
)
print(f'MLP parameters: {mlp_model.count_parameters():,}')

In [ ]:
trainer_cfg = TrainerConfig(
    epochs         = train_cfg['epochs'],
    lr             = train_cfg['lr'],
    weight_decay   = train_cfg['weight_decay'],
    clip_grad_norm = train_cfg['clip_grad_norm'],
    checkpoint_dir = dl_cfg['checkpoint_dir'],
    save_every     = train_cfg['save_every'],
    device         = train_cfg['device'],
)

trainer = Trainer(mlp_model, train_loader, val_loader, trainer_cfg)
history = trainer.fit()

In [ ]:
# ── Training curves ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'],   label='val')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(history['train_acc'], label='train')
axes[1].plot(history['val_acc'],   label='val')
axes[1].set_title('Accuracy'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── Evaluation on test set ─────────────────────────────────────────────────
device = trainer_cfg.device
report_df = classification_report_df(mlp_model, test_loader, dataset_mlp.idx2label, device)
print('Test-set classification report:')
print(report_df.round(3))

In [ ]:
# ── Confusion matrix ───────────────────────────────────────────────────────
confusion_matrix_plot(mlp_model, test_loader, dataset_mlp.idx2label, device)

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Model 2: ChordTransformer — sequence approach
# ═══════════════════════════════════════════════════════════
# Uses 'top_chords' column (list of (roman, count) tuples stored as string)
# A future improvement: store the full chord sequence in the DataFrame.

t_cfg = dl_cfg['transformer']

dataset_seq = HarmonicDataset(
    df_filtered,
    label_col   = 'primary_genre',
    mode        = 'sequence',
    seq_col     = 'top_chords',
    max_seq_len = t_cfg['max_seq_len'],
)
print(f'Sequence dataset: vocab_size={dataset_seq.vocab_size}')

idx_tr, idx_v, idx_te = dataset_seq.split(
    val_frac=train_cfg['val_frac'], test_frac=train_cfg['test_frac'], seed=train_cfg['seed']
)

tr_loader_seq = DataLoader(Subset(dataset_seq, idx_tr),  batch_size=32, shuffle=True)
va_loader_seq = DataLoader(Subset(dataset_seq, idx_v),   batch_size=32, shuffle=False)
te_loader_seq = DataLoader(Subset(dataset_seq, idx_te),  batch_size=32, shuffle=False)

transformer_model = ChordTransformer(
    vocab_size      = dataset_seq.vocab_size,
    num_classes     = dataset_seq.num_classes,
    mode            = 'classify',
    d_model         = t_cfg['d_model'],
    nhead           = t_cfg['nhead'],
    num_layers      = t_cfg['num_layers'],
    dim_feedforward = t_cfg['dim_feedforward'],
    max_seq_len     = t_cfg['max_seq_len'],
    dropout         = t_cfg['dropout'],
)
print(f'Transformer parameters: {transformer_model.count_parameters():,}')

In [ ]:
trainer_seq = Trainer(transformer_model, tr_loader_seq, va_loader_seq, trainer_cfg)
history_seq = trainer_seq.fit()